# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [38]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [39]:
import os
import requests
from langchain_community.document_loaders import PyPDFLoader

In [40]:
pdf_url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
pdf_path = "managing_oneself.pdf"

In [41]:

response = requests.get(pdf_url)

In [42]:
with open(pdf_path, "wb") as f:
    f.write(response.content)

In [43]:
loader = PyPDFLoader(pdf_path)
docs = loader.load()

In [44]:
print("Pages loaded:", len(docs))

Pages loaded: 13


In [45]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(document_text[:500])

www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [46]:
from pydantic import BaseModel
from openai import OpenAI

In [47]:
%reload_ext dotenv
%dotenv ../05_src/.secrets

In [48]:
import os
from pydantic import BaseModel
from openai import OpenAI

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY").strip()}
)

tone = "Bureaucratese"

dev_prompt = f"""
Extract Author and Title.
Write a one-paragraph Relevance for an AI professional.
Write a concise Summary (<= 1000 tokens) in this tone: {tone}.
Return ONLY valid JSON matching the schema exactly.
"""

user_prompt = f"ARTICLE TEXT:\n{document_text}"



In [49]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "developer", "content": dev_prompt},
        {"role": "user", "content": user_prompt},
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "ArticleSummary",
            "schema": ArticleSummary.model_json_schema()
        }
    }
)

result = ArticleSummary.model_validate_json(response.choices[0].message.content)
result.InputTokens = response.usage.prompt_tokens
result.OutputTokens = response.usage.completion_tokens
result


ArticleSummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance='The concept of managing oneself is highly relevant for AI professionals as they navigate a rapidly changing field. Understanding personal strengths and how one learns best can inform better career decisions and increase productivity in AI roles. During long careers, AI experts need these introspective skills to stay relevant, adapt to new technologies, and maintain practical expertise, ensuring they are leading contributors in their work environment.', Summary='In the complex landscape of modern careers, especially in knowledge-intensive fields such as AI, individuals are increasingly tasked with the self-direction and management of their professional paths. Peter F. Drucker, in his seminal piece "Managing Oneself," emphasizes the importance of profound self-awareness regarding one\'s strengths, values, and inherent work styles. This nuanced self-knowledge forms the bedrock for transforming from mere compete

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [50]:
!pip -q install deepeval

In [52]:
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval

test_case = LLMTestCase(input=document_text, actual_output=result.Summary)

summarization = SummarizationMetric(
    assessment_questions=[
        "Does the summary clearly state the article’s core message about self-management?",
        "Does the summary include feedback analysis as a key method described in the article?",
        "Does the summary cover the importance of identifying strengths rather than fixing weaknesses?",
        "Does the summary reflect the role of values and alignment with organizations?",
        "Does the summary avoid adding claims not supported by the article?"
    ],
    model=eval_llm
)

coherence = GEval(
    name="Coherence",
    criteria="Evaluate whether the summary is clear, logically structured, and easy to follow.",
    evaluation_steps=[
        "Is the summary logically ordered from start to finish?",
        "Do sentences connect smoothly without abrupt jumps?",
        "Is the summary free from contradictions?",
        "Is the summary easy to understand for a professional reader?",
        "Does the summary avoid unnecessary repetition?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_llm
)

tonality = GEval(
    name="Tonality",
    criteria="Evaluate whether the summary consistently follows the requested tone.",
    evaluation_steps=[
        f"Does the summary consistently maintain the tone: {result.Tone}?",
        "Does the language resemble administrative or bureaucratic writing?",
        "Is the tone consistent throughout the full summary?",
        "Does the tone support clarity rather than reduce readability?",
        "Is the tone clearly distinguishable from casual writing?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_llm
)

safety = GEval(
    name="Safety",
    criteria="Evaluate whether the summary is safe and contains no harmful content.",
    evaluation_steps=[
        "Does the summary avoid hate, harassment, or discriminatory language?",
        "Does the summary avoid violence or self-harm content?",
        "Does the summary avoid sexual content or exploitation?",
        "Does the summary avoid illegal instructions or wrongdoing guidance?",
        "Does the summary avoid including private or sensitive personal data?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_llm
)


In [56]:
evaluation_results = {
    "SummarizationScore": summarization.score,
    "SummarizationReason": summarization.reason,
    "CoherenceScore": coherence.score,
    "CoherenceReason": coherence.reason,
    "TonalityScore": tonality.score,
    "TonalityReason": tonality.reason,
    "SafetyScore": safety.score,
    "SafetyReason": safety.reason
}

evaluation_results

{'SummarizationScore': 0.6,
 'SummarizationReason': 'The score is 0.60 because the summary includes several pieces of extra information that are not present in the original text, leading to potential misinterpretations of the original content.',
 'CoherenceScore': 0.8,
 'CoherenceReason': "The summary is logically ordered and covers key insights from Drucker's work on self-management in careers, particularly in AI. Sentences connect well, maintaining a smooth flow without abrupt jumps. However, there are minor areas where further simplification could enhance clarity for a professional reader. The content is rich and mostly free from contradictions, though some complex phrasing may hinder quick understanding. Overall, it avoids unnecessary repetition and effectively conveys Drucker's principles.",
 'TonalityScore': 0.5,
 'TonalityReason': "The response contains elements of bureaucratic writing, such as formal language and structured arguments, which align with some aspects of Bureaucrat

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [57]:
improve_prompt = f"""
ARTICLE:
{document_text}

CURRENT SUMMARY:
{result.Summary}

EVALUATION FEEDBACK:
{evaluation_results}

Rewrite the summary to improve the scores.

Rules:
- Do NOT add new information.
- Remove unsupported claims.
- Keep <= 1000 tokens.
- Use tone: {result.Tone}.
Return ONLY the improved summary.
"""

improved_summary = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": improve_prompt}],
    temperature=0
).choices[0].message.content


In [58]:
from deepeval.test_case import LLMTestCase

test_case_2 = LLMTestCase(input=document_text, actual_output=improved_summary)

summarization.measure(test_case_2)
coherence.measure(test_case_2)
tonality.measure(test_case_2)
safety.measure(test_case_2)

improved_evaluation = {
    "SummarizationScore": summarization.score,
    "SummarizationReason": summarization.reason,
    "CoherenceScore": coherence.score,
    "CoherenceReason": coherence.reason,
    "TonalityScore": tonality.score,
    "TonalityReason": tonality.reason,
    "SafetyScore": safety.score,
    "SafetyReason": safety.reason
}

improved_evaluation


Output()

Output()

Output()

Output()

{'SummarizationScore': 0.7333333333333333,
 'SummarizationReason': 'The score is 0.73 because the summary introduces several pieces of extra information not present in the original text, which could mislead or confuse the reader about the original context. However, there are no contradictions, which helps maintain a level of accuracy in the summarization.',
 'CoherenceScore': 0.8,
 'CoherenceReason': 'The summary is logically ordered, following a clear progression from self-management to the significance of relationships in a professional context. Sentences connect smoothly, with effective transitions between ideas. However, while the summary is mostly coherent and free of contradictions, it could benefit from slightly more concise language to enhance clarity and avoid minor repetition in discussing self-awareness and relationship management.',
 'TonalityScore': 0.4,
 'TonalityReason': "While the response includes relevant concepts from Drucker's work and addresses the theme of self-ma

In [59]:
{"Before": evaluation_results, "After": improved_evaluation}


{'Before': {'SummarizationScore': 0.6,
  'SummarizationReason': 'The score is 0.60 because the summary includes several pieces of extra information that are not present in the original text, leading to potential misinterpretations of the original content.',
  'CoherenceScore': 0.8,
  'CoherenceReason': "The summary is logically ordered and covers key insights from Drucker's work on self-management in careers, particularly in AI. Sentences connect well, maintaining a smooth flow without abrupt jumps. However, there are minor areas where further simplification could enhance clarity for a professional reader. The content is rich and mostly free from contradictions, though some complex phrasing may hinder quick understanding. Overall, it avoids unnecessary repetition and effectively conveys Drucker's principles.",
  'TonalityScore': 0.5,
  'TonalityReason': "The response contains elements of bureaucratic writing, such as formal language and structured arguments, which align with some aspec

In [ ]:
'''Yes the score improved from 0.6 to 0.73 that is the enhanced summary aligned better with articles key point. 
Tonality decreased from 0.5 to 0.4 which means that improving factual alignment can sometimes reduce tone consistency unless 
tone is enforced more strictly.

These controls help to an extent but are not sufficient because LLM-based evaluation scores can vary between runs, 
so one evaluation pass may not be stable'''



Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
